# Solution: Pareto Optimization of SEI Additives with ALCHEMI

Lithium-metal and lithium-ion batteries depend on interfaces. During cycling, electrolyte molecules touch highly reactive electrode surfaces: the negative electrode can drive reduction chemistry, while the positive electrode can drive oxidation chemistry. Some decomposition is harmful because it consumes electrolyte and active lithium, but some early decomposition can be useful when it forms a thin passivating interphase. For battery context, see the SEI modeling review by [Shi et al.](https://www.nature.com/articles/s41524-018-0064-0).

At the anode, this protective layer is usually called the solid electrolyte interphase (SEI). A useful SEI should block electrons, allow Li+ transport, remain chemically and mechanically stable, and avoid continuously attacking the electrolyte; this electronically insulating, ionically conducting passivation picture is reviewed by [Shi et al.](https://www.nature.com/articles/s41524-018-0064-0). Electrolyte additives are often chosen because they react before the bulk solvent and help seed a more protective SEI; additive design and preferential film formation are reviewed by [Balakrishnan et al.](https://www.sciencedirect.com/science/article/pii/S2451910320300089). The design tension is subtle: an additive that barely interacts may do nothing, while one that binds or decomposes too aggressively may create impedance, gas, or unstable products.

A realistic battery-interface simulation would need explicit liquid electrolyte, electron transfer, lithium-ion motion, voltage, many reaction pathways, long timescales, and multiple surface structures. The breadth of these modeling challenges is a central theme in the SEI modeling literature ([Shi et al.](https://www.nature.com/articles/s41524-018-0064-0)). Instead, this notebook demonstrates a small, transparent screening proxy using the skills from Part 1: generate surface+molecule systems, relax them in ALCHEMI Toolkit batches, compute binding energies, and use those energies to rank candidates.

The two challenge objectives encode the tradeoff. First, a molecule should have useful moderate interaction with a reactive Li-metal proxy, which represents SEI seeding. Second, once a passivating SEI-like product exists, the molecule should interact weakly with that surface, which represents compatibility with a protective layer. Because these goals can conflict, this solution uses Pareto hypervolume improvement rather than a single hand-picked threshold; the hypervolume indicator is a standard multi-objective quality measure ([Hypervolume bibliography](https://hypervolume.org/bibliography.html)).

The chemistry here is intentionally simplified. Li metal is a reactive anode proxy. Each molecule class maps to one passivating SEI-product proxy surface using the lookup table in `data/class_surface_lookup.csv`. The bundled structures are a starter panel for a workflow exercise, not production battery-interface reference models. If `data/custom_molecule_manifest.csv` exists, this solution appends those literature/custom candidates before running the workflow; custom rows should record citation/provenance. For example, FEC is included because it is a widely studied SEI additive with reported LiF-containing reduction products ([PNNL summary](https://www.pnnl.gov/publications/reduction-mechanism-fluoroethylene-carbonate-stable-solid-electrolyte-interphase-film)).

The reward functions are deliberately simple but no longer tied to one arbitrary target energy. They use a moderate-adsorption window for Li-metal seeding and a weak-adsorption preference for SEI passivation, following the same qualitative logic used in SEI-additive and Sabatier-style surface-screening literature ([Lee et al.](https://www.frontiersin.org/journals/energy-research/articles/10.3389/fenrg.2021.654460/full)). The exact constants remain a challenge calibration so the task is reproducible and model-free to grade.


## Solution Outputs

This notebook writes `outputs/challenge_submission.csv` and `outputs/raw_component_energies.csv`. The separate grader can check both files without running model calls.


## Control Panel

These defaults mirror the Part 1 tutorial but keep the challenge small. You may reduce `TOOLKIT_N_STEPS` while debugging, then rerun with the default before submitting.


In [ ]:
from pathlib import Path

TOOLKIT_CHECKPOINT = "medium-mpa-0"
TOOLKIT_HEAD = None
TOOLKIT_DEVICE = "auto"
TOOLKIT_DTYPE = "float32"
TOOLKIT_COMPILE_MODEL = False
TOOLKIT_ENABLE_CUEQ = True
TOOLKIT_DT = 0.01
TOOLKIT_N_STEPS = 300
TOOLKIT_FMAX = 0.10
TOOLKIT_D3BJ = None
BATCH_SIZE = 4

ADSORPTION_HEIGHT_A = 2.6
FROZEN_SURFACE_FRACTION = 0.5

OUTPUT_DIR = Path("outputs")
SUBMISSION_PATH = OUTPUT_DIR / "challenge_submission.csv"
RAW_COMPONENT_ENERGIES_PATH = OUTPUT_DIR / "raw_component_energies.csv"


## Setup

The [ALCHEMI Toolkit documentation](https://nvidia.github.io/nvalchemi-toolkit/) describes the same core workflow used in Part 1: structures are represented as `AtomicData`, packed into `Batch` objects, evaluated by model wrappers such as MACE, and relaxed with Toolkit dynamics/optimizer components. MACE is a higher-order equivariant message-passing MLIP architecture ([Batatia et al., 2022](https://openreview.net/forum?id=YPpSngE-ZU)), and this notebook uses ASE `Atoms` objects for local structure handling ([Larsen et al., 2017](https://doi.org/10.1088/1361-648X/aa680e)). This notebook reuses the Part 1 helper backend so your challenge code focuses on the scientific workflow and bookkeeping.


In [ ]:
import os
import sys
from importlib.metadata import version, PackageNotFoundError

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "data" / "molecule_manifest.csv").exists():
    candidate = NOTEBOOK_DIR / "challenge-sei"
    if (candidate / "data" / "molecule_manifest.csv").exists():
        NOTEBOOK_DIR = candidate.resolve()
    else:
        raise RuntimeError("Start Jupyter from challenge-sei or from the repository root.")
os.chdir(NOTEBOOK_DIR)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

# The solution notebook is intended for instructor reruns. Allow the Part 1
# cache helper to refresh the same solution cache labels when cells are rerun.
os.environ["ALCHEMI_ALLOW_CACHE_OVERWRITE"] = "1"

REPO_ROOT = NOTEBOOK_DIR.parent
PART1_ROOT = REPO_ROOT / "part-1-batched-adsorption"
if not (PART1_ROOT / "helpers" / "__init__.py").exists():
    raise RuntimeError("Cannot find Part 1 helpers. Keep challenge-sei beside part-1-batched-adsorption.")
sys.path.insert(0, str(PART1_ROOT))

import numpy as np
import pandas as pd
from ase.io import read as ase_read
from ase.io import write as ase_write

from helpers import (
    ToolkitRelaxationConfig,
    ToolkitD3BJConfig,
    check_toolkit_native_api,
    get_toolkit_relaxation_engine,
    ase_to_atomic_data,
    atomic_data_to_ase,
    make_active_mask,
    display_widgets_grid,
)
from challenge_utils.pareto import dominates, hypervolume_2d, pareto_flags
from challenge_utils.rewards import passivation_score, seeding_score

print(f"Challenge folder : {NOTEBOOK_DIR.name}")
print(f"Part 1 helpers   : {PART1_ROOT.relative_to(REPO_ROOT)}")
for pkg in ("ase", "numpy", "pandas", "torch", "nvalchemi-toolkit", "ovito"):
    try:
        print(f"{pkg:<18}: {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:<18}: not installed")


## 1. Load The Challenge Manifests

Fill in this cell so every molecule has its class-specific passivating surface. Keep the baseline rows (`EC`, `EMC`) because they define the starting Pareto front. The starter panel can be extended with `data/custom_molecule_manifest.csv`; custom rows should include source/provenance from additive or SEI literature. Good starting points include broad additive reviews such as [Xu et al.](https://www.sciencedirect.com/science/article/pii/S0378775306017538) and [Balakrishnan et al.](https://www.sciencedirect.com/science/article/pii/S2451910320300089), Li-metal SEI reviews such as [Li et al.](https://pmc.ncbi.nlm.nih.gov/articles/PMC5063117/), focused additive papers on [VC-derived SEI](https://www.sciencedirect.com/science/article/abs/pii/S1572665722001187) and [FEC reduction products](https://www.pnnl.gov/publications/reduction-mechanism-fluoroethylene-carbonate-stable-solid-electrolyte-interphase-film), and borate-salt additive studies such as [LiDFOB/LiBOB SEI modulation](https://www.sciencedirect.com/science/article/pii/S016943321830357X).


In [ ]:
molecules_df = pd.read_csv("data/molecule_manifest.csv")
custom_manifest_path = Path("data/custom_molecule_manifest.csv")
if custom_manifest_path.exists():
    custom_molecules_df = pd.read_csv(custom_manifest_path)
    molecules_df = pd.concat([molecules_df, custom_molecules_df], ignore_index=True)
    print(f"Loaded {len(custom_molecules_df)} custom/literature molecule row(s).")
else:
    print("No data/custom_molecule_manifest.csv found; using the starter molecule panel.")
if molecules_df["candidate_id"].duplicated().any():
    duplicated = sorted(molecules_df.loc[molecules_df["candidate_id"].duplicated(), "candidate_id"].unique())
    raise RuntimeError(f"Duplicate candidate_id value(s): {duplicated}")
surfaces_df = pd.read_csv("data/surface_manifest.csv")
lookup_df = pd.read_csv("data/class_surface_lookup.csv")

challenge_df = molecules_df.merge(
    lookup_df[["molecule_class", "passivating_surface_id"]],
    on="molecule_class",
    how="left",
    validate="many_to_one",
)
if challenge_df["passivating_surface_id"].isna().any():
    missing = challenge_df.loc[
        challenge_df["passivating_surface_id"].isna(),
        ["candidate_id", "molecule_class"],
    ]
    raise RuntimeError(f"Missing passivating-surface lookup rows:\n{missing}")

assert set(challenge_df["role"]) == {"baseline", "additive"}
assert {"EC", "EMC"}.issubset(set(challenge_df.loc[challenge_df["role"].eq("baseline"), "candidate_id"]))

display(challenge_df[["candidate_id", "role", "molecule_class", "passivating_surface_id", "structure_path"]])
display(surfaces_df[["surface_id", "role", "structure_path"]])


## 2. Build The Toolkit Relaxation Engine

This is the same native Toolkit path used in Part 1. ALCHEMI exposes batched atomistic data, MLIP model wrappers, hooks, and dynamics components for workflows like geometry optimization ([ALCHEMI Toolkit docs](https://nvidia.github.io/nvalchemi-toolkit/)). D3 is disabled by default for a compact challenge run; if you enable it for an instructor rerun, record the parameters in your notes.


In [ ]:
status = check_toolkit_native_api()
print(status["message"])
if not status["available"]:
    raise RuntimeError("ALCHEMI Toolkit native API is not available in this kernel.")

if isinstance(TOOLKIT_D3BJ, dict):
    TOOLKIT_D3BJ = ToolkitD3BJConfig(**TOOLKIT_D3BJ)

relaxation_config = ToolkitRelaxationConfig(
    name="toolkit",
    cache_dir=(OUTPUT_DIR / "cache_json").as_posix(),
    use_cached_responses=False,
    toolkit_checkpoint=TOOLKIT_CHECKPOINT,
    toolkit_head=TOOLKIT_HEAD,
    toolkit_device=TOOLKIT_DEVICE,
    toolkit_dtype=TOOLKIT_DTYPE,
    toolkit_compile_model=TOOLKIT_COMPILE_MODEL,
    toolkit_enable_cueq=TOOLKIT_ENABLE_CUEQ,
    toolkit_dt=TOOLKIT_DT,
    toolkit_n_steps=TOOLKIT_N_STEPS,
    toolkit_fmax=TOOLKIT_FMAX,
    toolkit_d3bj=TOOLKIT_D3BJ,
    toolkit_require_d3bj=TOOLKIT_D3BJ is not None,
)
RELAXATION_ENGINE = get_toolkit_relaxation_engine(relaxation_config)
print(f"Toolkit relaxation engine ready: {RELAXATION_ENGINE.name}")


## 3. Structure Helpers

These helpers load the bundled structures and place each molecule above the center of a teaching slab using ASE-style structure manipulation ([Larsen et al., 2017](https://doi.org/10.1088/1361-648X/aa680e)). The placement is intentionally minimal: the challenge is about reproducing the Part 1 workflow logic, not building a production adsorption-site search.


In [ ]:
def load_atoms(relative_path):
    """Read an ASE Atoms object from a path relative to the challenge folder."""
    atoms = ase_read(Path(relative_path))
    return atoms


def gas_box(atoms, *, box_A=15.0):
    """Return a centered molecule in a periodic cubic vacuum box."""
    gas = atoms.copy()
    gas.set_cell([box_A, box_A, box_A])
    gas.set_pbc([True, True, True])
    gas.center()
    return gas


def surface_center_xy(surface):
    """Return the in-plane center of the first two periodic cell vectors."""
    cell = np.asarray(surface.cell.array, dtype=float)
    center = 0.5 * (cell[0] + cell[1])
    return center[:2]


def place_molecule_on_surface(surface, molecule, *, height_A=ADSORPTION_HEIGHT_A):
    """Place a molecule above the slab center at a fixed height from the top layer."""
    slab = surface.copy()
    mol = molecule.copy()
    mol.translate(-mol.get_center_of_mass())
    top_z = float(np.max(slab.positions[:, 2]))
    bottom_z = float(np.min(mol.positions[:, 2]))
    xy = surface_center_xy(slab)
    mol.translate([xy[0], xy[1], top_z + height_A - bottom_z])
    combined = slab + mol
    combined.set_cell(slab.cell)
    combined.set_pbc(slab.pbc)
    return combined


def combined_active_mask(surface, combined):
    """Freeze the lower slab fraction and leave the adsorbed molecule active."""
    surface_mask = make_active_mask(surface, bottom_fraction=FROZEN_SURFACE_FRACTION)
    return surface_mask + [True] * (len(combined) - len(surface))


def relax_structures(jobs, *, batch_size=BATCH_SIZE, label_prefix="sei_challenge"):
    """Relax job dictionaries in Toolkit batches and return energy/result rows."""
    rows = []
    for start in range(0, len(jobs), batch_size):
        chunk = jobs[start:start + batch_size]
        payloads = [
            ase_to_atomic_data(job["atoms"], structure_id=job["job_id"], active_mask=job.get("active_mask"))
            for job in chunk
        ]
        reply = RELAXATION_ENGINE.relax(payloads, label=f"{label_prefix}_{start // batch_size + 1:03d}")
        for job, result in zip(chunk, reply.atoms):
            rows.append({
                **{key: value for key, value in job.items() if key not in {"atoms", "active_mask"}},
                "energy_eV": float(result.energy),
                "converged": bool(result.converged),
                "optimizer_nsteps": int(result.num_optimization_steps),
                "relaxed_atoms": atomic_data_to_ase(result),
            })
    return rows


## 4. Build And Relax The Jobs

You need three groups of energies, following the component-energy bookkeeping common to adsorption-style screening workflows. The relaxation path uses the batched ALCHEMI Toolkit pattern described in the [Toolkit documentation](https://nvidia.github.io/nvalchemi-toolkit/).

1. Gas molecules: `E_species`.
2. Clean surfaces: `E_surface` for `Li_metal` and every passivating surface used by the candidates.
3. Combined systems: `E_surface+species` for molecule on Li metal and molecule on its class-specific passivating surface.


In [ ]:
molecule_atoms = {
    row.candidate_id: load_atoms(row.structure_path)
    for row in challenge_df.itertuples(index=False)
}
surface_atoms = {
    row.surface_id: load_atoms(row.structure_path)
    for row in surfaces_df.itertuples(index=False)
}
surface_meta = surfaces_df.set_index("surface_id")

used_surface_ids = sorted({"Li_metal", *challenge_df["passivating_surface_id"].unique()})

# Gas references: one isolated molecule per candidate.
gas_jobs = []
for row in challenge_df.itertuples(index=False):
    atoms = gas_box(molecule_atoms[row.candidate_id])
    atoms.info["charge"] = int(row.charge)
    atoms.info["mult"] = int(row.multiplicity)
    gas_jobs.append({
        "job_id": f"gas_{row.candidate_id}",
        "candidate_id": row.candidate_id,
        "interaction": "gas",
        "surface_id": "",
        "atoms": atoms,
    })

# Clean-surface references: Li metal plus every class-specific SEI proxy used here.
clean_surface_jobs = []
for surface_id in used_surface_ids:
    atoms = surface_atoms[surface_id].copy()
    atoms.info["charge"] = int(surface_meta.loc[surface_id, "charge"])
    atoms.info["mult"] = int(surface_meta.loc[surface_id, "multiplicity"])
    clean_surface_jobs.append({
        "job_id": f"clean_{surface_id}",
        "surface_id": surface_id,
        "interaction": "clean_surface",
        "atoms": atoms,
        "active_mask": make_active_mask(atoms, bottom_fraction=FROZEN_SURFACE_FRACTION),
    })

# Combined systems: each molecule on Li metal and on its mapped passivating surface.
combined_jobs = []
for row in challenge_df.itertuples(index=False):
    for interaction, surface_id in (
        ("li_metal", "Li_metal"),
        ("passivating", row.passivating_surface_id),
    ):
        surface = surface_atoms[surface_id].copy()
        molecule = molecule_atoms[row.candidate_id].copy()
        combined = place_molecule_on_surface(surface, molecule)
        combined.info["charge"] = int(row.charge) + int(surface_meta.loc[surface_id, "charge"])
        combined.info["mult"] = 1
        combined_jobs.append({
            "job_id": f"combined_{interaction}_{row.candidate_id}_{surface_id}",
            "candidate_id": row.candidate_id,
            "interaction": interaction,
            "surface_id": surface_id,
            "atoms": combined,
            "active_mask": combined_active_mask(surface, combined),
        })

print(f"Gas jobs           : {len(gas_jobs)}")
print(f"Clean-surface jobs : {len(clean_surface_jobs)}")
print(f"Combined jobs      : {len(combined_jobs)}")

OUTPUT_DIR.mkdir(exist_ok=True)
gas_results = relax_structures(gas_jobs, label_prefix="solution_sei_gas")
clean_surface_results = relax_structures(clean_surface_jobs, label_prefix="solution_sei_clean_surface")
combined_results = relax_structures(combined_jobs, label_prefix="solution_sei_combined")

print("Relaxations complete.")


## 5. Compute Binding Energies

Use the same adsorption-energy convention as Part 1:

`E_bind = E_surface+species - E_surface - E_species`

Negative values mean exothermic binding in this challenge convention. The isolated gas molecule is a controlled computational reference, not a full solution-phase free energy; solution SEI modeling would need explicit electrolyte, electrode potential, Li+ coordination, and sampling corrections ([Shi et al.](https://www.nature.com/articles/s41524-018-0064-0)). The broad adsorption-strength interpretation used later is guided by adsorption thermodynamics and energy-scale discussions such as [Aich et al.](https://link.springer.com/article/10.1007/s11696-025-04218-x).


In [ ]:
gas_energy = {
    row["candidate_id"]: float(row["energy_eV"])
    for row in gas_results
}
surface_energy = {
    row["surface_id"]: float(row["energy_eV"])
    for row in clean_surface_results
}
combined_energy = {
    (row["candidate_id"], row["interaction"]): float(row["energy_eV"])
    for row in combined_results
}
combined_surface = {
    (row["candidate_id"], row["interaction"]): row["surface_id"]
    for row in combined_results
}

raw_rows = []
for row in challenge_df.itertuples(index=False):
    for interaction, surface_id in (
        ("li_metal", "Li_metal"),
        ("passivating", row.passivating_surface_id),
    ):
        actual_surface_id = combined_surface[(row.candidate_id, interaction)]
        if actual_surface_id != surface_id:
            raise RuntimeError(
                f"Unexpected surface for {row.candidate_id}/{interaction}: "
                f"{actual_surface_id} != {surface_id}"
            )
        raw_rows.append({
            "candidate_id": row.candidate_id,
            "interaction": interaction,
            "surface_id": surface_id,
            "E_surface_species_eV": combined_energy[(row.candidate_id, interaction)],
            "E_surface_eV": surface_energy[surface_id],
            "E_species_eV": gas_energy[row.candidate_id],
        })

raw_component_energies_df = pd.DataFrame(raw_rows)
raw_component_energies_df.to_csv(RAW_COMPONENT_ENERGIES_PATH, index=False)
print(f"Wrote {RAW_COMPONENT_ENERGIES_PATH}")
display(raw_component_energies_df.head())


In [ ]:
binding_rows = []
raw_by_key = raw_component_energies_df.set_index(["candidate_id", "interaction"])
for row in challenge_df.itertuples(index=False):
    li = raw_by_key.loc[(row.candidate_id, "li_metal")]
    pas = raw_by_key.loc[(row.candidate_id, "passivating")]
    binding_rows.append({
        "candidate_id": row.candidate_id,
        "role": row.role,
        "molecule_class": row.molecule_class,
        "passivating_surface_id": row.passivating_surface_id,
        "E_bind_Li_eV": float(li["E_surface_species_eV"] - li["E_surface_eV"] - li["E_species_eV"]),
        "E_bind_passivating_eV": float(pas["E_surface_species_eV"] - pas["E_surface_eV"] - pas["E_species_eV"]),
    })

binding_df = pd.DataFrame(binding_rows)
display(binding_df[["candidate_id", "role", "E_bind_Li_eV", "E_bind_passivating_eV"]])


## 5b. Visual Inspect Relaxed Adsorption Geometries With OVITO

Before turning relaxed energies into scores, inspect the relaxed surface+molecule geometries. This is where OVITO is most useful: it can catch obvious bad placements, detached fragments, molecules crossing periodic boundaries, or suspicious relaxed geometries before those structures enter the Pareto analysis. The cell writes every relaxed combined structure to `outputs/ovito_structures/` as `extxyz`, then tries to show a small OVITO widget grid in the notebook. If the OVITO widget stack is unavailable in your environment, open the saved `extxyz` files directly in OVITO. See the [OVITO Python documentation](https://www.ovito.org/docs/current/python/) and Stukowski's OVITO paper ([doi:10.1088/0965-0393/18/1/015012](https://doi.org/10.1088/0965-0393/18/1/015012)).


In [ ]:
OVITO_STRUCTURE_DIR = OUTPUT_DIR / "ovito_structures"


def safe_structure_name(*parts):
    """Return a filesystem-safe structure stem from metadata fields."""
    text = "_".join(str(part) for part in parts if str(part))
    return "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in text)


def write_ovito_inspection_structures(results, *, output_dir=OVITO_STRUCTURE_DIR):
    """Write relaxed combined structures as EXTXYZ files for OVITO inspection."""
    output_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for result in results:
        if result.get("interaction") not in {"li_metal", "passivating"}:
            continue
        candidate_id = result["candidate_id"]
        interaction = result["interaction"]
        surface_id = result["surface_id"]
        atoms = result["relaxed_atoms"].copy()
        atoms.info["candidate_id"] = candidate_id
        atoms.info["interaction"] = interaction
        atoms.info["surface_id"] = surface_id
        atoms.info["energy_eV"] = float(result["energy_eV"])
        path = output_dir / f"{safe_structure_name(candidate_id, interaction, surface_id)}.extxyz"
        ase_write(path, atoms, format="extxyz")
        rows.append({
            "candidate_id": candidate_id,
            "interaction": interaction,
            "surface_id": surface_id,
            "energy_eV": float(result["energy_eV"]),
            "converged": bool(result["converged"]),
            "structure_path": path.as_posix(),
        })
    if not rows:
        raise RuntimeError("No relaxed combined structures were found in combined_results.")
    return pd.DataFrame(rows).sort_values(["candidate_id", "interaction"]).reset_index(drop=True)


def choose_inspection_candidates(binding_table, *, max_candidates=6):
    """Choose baseline rows plus binding-energy outliers for visual inspection."""
    baseline_ids = binding_table.loc[
        binding_table["role"].eq("baseline"), "candidate_id"
    ].tolist()
    strongest_li_ids = binding_table.sort_values("E_bind_Li_eV").head(2)["candidate_id"].tolist()
    weakest_passivating_ids = binding_table.sort_values(
        "E_bind_passivating_eV", ascending=False
    ).head(2)["candidate_id"].tolist()
    ordered = [*baseline_ids, *strongest_li_ids, *weakest_passivating_ids]
    return list(dict.fromkeys(ordered))[:max_candidates]


inspection_df = write_ovito_inspection_structures(combined_results)
INSPECT_CANDIDATE_IDS = choose_inspection_candidates(binding_df)
print(f"Wrote {len(inspection_df)} OVITO structure file(s) to {OVITO_STRUCTURE_DIR}")
print("Inspecting:", ", ".join(INSPECT_CANDIDATE_IDS))
display(inspection_df[inspection_df["candidate_id"].isin(INSPECT_CANDIDATE_IDS)])

widget_rows = []
for candidate_id in INSPECT_CANDIDATE_IDS:
    row_widgets = []
    for interaction in ("li_metal", "passivating"):
        matches = inspection_df[
            inspection_df["candidate_id"].eq(candidate_id)
            & inspection_df["interaction"].eq(interaction)
        ]
        if matches.empty:
            continue
        row = matches.iloc[0]
        label = f"{candidate_id} | {row.interaction} | {row.surface_id}"
        row_widgets.append((label, row.structure_path))
    if row_widgets:
        widget_rows.append(row_widgets)

try:
    display_widgets_grid(widget_rows, width="390px", height="310px", show_cell=True)
except Exception as exc:
    print(f"OVITO widget display unavailable: {type(exc).__name__}: {exc}")
    print("Open these EXTXYZ files directly in OVITO instead:")
    display(inspection_df[["candidate_id", "interaction", "surface_id", "structure_path"]])


## 6. Compute Literature-Motivated Reward Scores

The reward functions are provided in `challenge_utils.rewards` so everyone uses the same challenge rubric. They convert each binding energy into an adsorption-strength magnitude, `strength = max(0, -E_bind)`, then apply two bounded rewards.

The Li-metal seeding reward follows a Sabatier-style idea from surface chemistry: useful interaction should be neither too weak nor too strong. It gives full reward for moderate adsorption strengths from 0.8 to 1.5 eV, tapers to zero below 0.5 eV, and tapers to zero above 2.0 eV. This replaces the earlier single `-1 eV` target with a broader moderate-chemisorption window.

The passivation reward follows the SEI picture that a dense, intact interphase should suppress continued electrolyte reduction. It gives full reward for weak or endothermic adsorption on the SEI proxy, `strength <= 0.3 eV`, and reaches zero by `0.8 eV`.

These constants are a transparent screening calibration, not universal battery chemistry. They are not fitted to a validated SEI dataset. The bounds are chosen by mapping broad literature ideas onto this small adsorption exercise: Balakrishnan et al. motivate the seeding objective because electrolyte additives can form protective electrode films before the main solvent continues reacting; Shi et al. motivate the passivation objective because a useful SEI should electronically block/passivate further electrolyte reduction; Lee et al. motivate the window shape because Sabatier-style surface descriptors reward binding that is neither too weak nor too strong; and adsorption-energy reviews give broad physical-adsorption versus chemical-adsorption energy scales in kJ/mol.

Using the standard conversion `1 eV per molecule = 96.485 kJ/mol`, the challenge bounds correspond approximately to `0.3 eV = 29 kJ/mol`, `0.5 eV = 48 kJ/mol`, `0.8 eV = 77 kJ/mol`, `1.5 eV = 145 kJ/mol`, and `2.0 eV = 193 kJ/mol`. In this rubric, `<=0.3 eV` is treated as weak adsorption suitable for an already-passivating SEI proxy; `0.8-1.5 eV` is treated as a moderate chemisorption-like window for Li-metal seeding; `<0.5 eV` is too weak to seed much chemistry; and `>2.0 eV` is penalized as overbinding.

Background links for the rubric: [electrolyte additives and protective films](https://www.sciencedirect.com/science/article/pii/S2451910320300089), [SEI passivation and electron blocking](https://www.nature.com/articles/s41524-018-0064-0), [Sabatier-style adsorption tradeoffs](https://www.frontiersin.org/journals/energy-research/articles/10.3389/fenrg.2021.654460/full), and [adsorption energy scales](https://link.springer.com/article/10.1007/s11696-025-04218-x).


In [ ]:
scored_df = binding_df.copy()
scored_df["seeding_score"] = scored_df["E_bind_Li_eV"].map(seeding_score)
scored_df["passivation_score"] = scored_df["E_bind_passivating_eV"].map(passivation_score)

display(scored_df[["candidate_id", "role", "seeding_score", "passivation_score"]])


## 7. Pareto Front And Hypervolume Improvement

Treat both scores as objectives to maximize. The baseline front is built from `EC` and `EMC`. For each additive, compute how much the 2D dominated hypervolume increases when that additive is added to the baseline front. Use reference point `(0, 0)`. Hypervolume is a standard Pareto-front quality indicator in multi-objective optimization; see the survey references collected at [hypervolume.org](https://hypervolume.org/bibliography.html), including Zitzler and Thiele's early work and later reviews.


In [ ]:
final_df = scored_df.copy()
points = list(zip(final_df["seeding_score"], final_df["passivation_score"]))
final_df["is_pareto"] = pareto_flags(points)

baseline_points = list(zip(
    final_df.loc[final_df["role"].eq("baseline"), "seeding_score"],
    final_df.loc[final_df["role"].eq("baseline"), "passivation_score"],
))
baseline_hv = hypervolume_2d(baseline_points)

improvements = []
for row in final_df.itertuples(index=False):
    if row.role == "baseline":
        improvements.append(0.0)
    else:
        point = (row.seeding_score, row.passivation_score)
        improvements.append(hypervolume_2d([*baseline_points, point]) - baseline_hv)
final_df["hypervolume_improvement"] = improvements

print(f"Baseline hypervolume: {baseline_hv:.4f}")
display(final_df.sort_values("hypervolume_improvement", ascending=False))


## 8. Select Your Additive And Submit

Mark exactly one additive as `selected=True`: the additive with the maximum hypervolume improvement. Baseline rows should not be selected.


In [ ]:
submission = final_df.copy()
additives = submission[submission["role"].eq("additive")].copy()
if additives.empty:
    raise RuntimeError("No additive rows are available to select.")

selected_id = (
    additives
    .sort_values(["hypervolume_improvement", "candidate_id"], ascending=[False, True])
    .iloc[0]["candidate_id"]
)
submission["selected"] = submission["candidate_id"].eq(selected_id)

print(f"Selected additive: {selected_id}")
display(
    submission[[
        "candidate_id", "role", "seeding_score", "passivation_score",
        "is_pareto", "hypervolume_improvement", "selected",
    ]].sort_values("hypervolume_improvement", ascending=False)
)


In [ ]:
required_columns = [
    "candidate_id", "role", "molecule_class", "passivating_surface_id",
    "E_bind_Li_eV", "E_bind_passivating_eV", "seeding_score",
    "passivation_score", "is_pareto", "hypervolume_improvement", "selected",
]
missing = [column for column in required_columns if column not in submission.columns]
if missing:
    raise RuntimeError(f"Submission is missing required columns: {missing}")
if int(submission["selected"].sum()) != 1:
    raise RuntimeError("Exactly one row must be selected.")

OUTPUT_DIR.mkdir(exist_ok=True)
submission[required_columns].to_csv(SUBMISSION_PATH, index=False)
print(f"Wrote {SUBMISSION_PATH}")
display(submission[required_columns])


## References And Further Reading

- NVIDIA [ALCHEMI Toolkit documentation](https://nvidia.github.io/nvalchemi-toolkit/) for `AtomicData`, `Batch`, model wrappers, and Toolkit dynamics.
- Batatia et al., [MACE: Higher Order Equivariant Message Passing Neural Networks for Fast and Accurate Force Fields](https://openreview.net/forum?id=YPpSngE-ZU), NeurIPS 2022.
- Larsen et al., [The Atomic Simulation Environment - a Python library for working with atoms](https://doi.org/10.1088/1361-648X/aa680e), J. Phys.: Condens. Matter 2017.
- Stukowski, [Visualization and analysis of atomistic simulation data with OVITO - the Open Visualization Tool](https://doi.org/10.1088/0965-0393/18/1/015012), Modelling Simul. Mater. Sci. Eng. 2010.
- Shi et al., [Review on modeling of the anode solid electrolyte interphase (SEI) for lithium-ion batteries](https://www.nature.com/articles/s41524-018-0064-0), npj Computational Materials 2018.
- Xu et al., [A review on electrolyte additives for lithium-ion batteries](https://www.sciencedirect.com/science/article/pii/S0378775306017538), J. Power Sources 2007.
- Balakrishnan et al., [Electrolyte additives for improved lithium-ion battery performance and overcharge protection](https://www.sciencedirect.com/science/article/pii/S2451910320300089), 2020.
- Li et al., [A Review of Solid Electrolyte Interphases on Lithium Metal Anode](https://pmc.ncbi.nlm.nih.gov/articles/PMC5063117/), Advanced Science 2016.
- [Insights into the efficient roles of solid electrolyte interphase derived from vinylene carbonate additive in rechargeable batteries](https://www.sciencedirect.com/science/article/abs/pii/S1572665722001187), 2022.
- [Modulation of solid electrolyte interphase of lithium-ion batteries by LiDFOB and LiBOB electrolyte additives](https://www.sciencedirect.com/science/article/pii/S016943321830357X), 2018.
- Zhang et al., [Reduction Mechanism of Fluoroethylene Carbonate for Stable Solid-Electrolyte Interphase Film on Silicon Anode](https://www.pnnl.gov/publications/reduction-mechanism-fluoroethylene-carbonate-stable-solid-electrolyte-interphase-film), ChemSusChem 2013.
- Lee et al., [The Sabatier Principle in Electrocatalysis: Basics, Limitations, and Extensions](https://www.frontiersin.org/journals/energy-research/articles/10.3389/fenrg.2021.654460/full), Frontiers in Energy Research 2021.
- Aich et al., [Determination of thermodynamic parameters in adsorption studies: a review](https://link.springer.com/article/10.1007/s11696-025-04218-x), Chemical Papers 2025.
- [Hypervolume bibliography](https://hypervolume.org/bibliography.html) for Pareto hypervolume indicator references.
